# Validación temporal cruzada MDLF — Estudio de ablación

**Baseline:** Trial #57, ganador DB 75mil (`mdlf_junio2025_v2_line`)  
**Diseño:** 7 condiciones × 3 folds temporales = 21 runs  
**Métricas:** Acc@1, Acc@5, F1-weighted mean ± std + 95% CI bootstrap  

| Orden | Condición | Cambio |
|-------|-----------|--------|
| 1 | Baseline (full) | — |
| 2 | −neighborhood | include_neighborhood_crowd=False |
| 3 | −prox | include_prox=False |
| 4 | SGD | optimizer='sgd' |
| 5 | −LDA | ablation={'lda'} |
| 6 | −LINE | ablation={'line'} |
| 7 | −LDA −LINE | ablation={'lda','line'} |
| 8 | −prox −neighborhood | include_prox=False, nbcrowd=False |
| 9 | −temporal | ablation={'temporal'} |
| 10 | −individual | ablation={'individual'} |
| 11 | −crowd | ablation={'crowd'} |
| 12 | −prox −temporal | ablation={'temporal'}, include_prox=False |

In [ ]:
import sys
import numpy as np
import pandas as pd
from datetime import date
from pathlib import Path
from sklearn.metrics import f1_score

_estudios = Path('/home/jupyter-lquijano/david/estudios')
_mdlf_dir = _estudios / 'MDLF'
sys.path.insert(0, str(_estudios))

from MDLF.pipeline_mdlf import PipelineMDLF

In [ ]:
# ── Rutas ───────────────────────────────────────────────────────────────────
ALMACEN   = '75mil_tarjetas_enero_junio_2025'
LINE_PATH = _mdlf_dir / 'runs' / 'line_fixed_optuna.parquet'
_descomp  = _estudios / 'descomprimido'
TTP_PATHS = [str(_descomp / f'TTP_PERIODO_0{i}_2025.csv') for i in range(1, 7)]
OD_MATRICES_SUBDIRS = [
    'validaciones_3_7_febrero', 'validaciones_8_9_febrero',
    'validaciones_10_16_febrero', 'validaciones_17_23_febrero',
    'validaciones_24febrero_2marzo',
]
HOLIDAY_DATES = frozenset([
    date(2025, 1, 1), date(2025, 1, 6), date(2025, 4, 17), date(2025, 4, 18),
    date(2025, 5, 1), date(2025, 5, 2), date(2025, 5, 15),
])
N_EPOCHS = 15

# ── Hiperparámetros ganadores — Trial #57 (75mil) ───────────────────────────
BEST_PARAMS = dict(
    n_topics_cercanias=48, lda_alpha=0.2122, lda_beta=0.0029,
    lda_log_counts=True, lda_min_df=5, lda_min_station_trips=0,
    lda_use_periods=False, lda_token_granularity='temporal',
    n_heads=2, d_t=8, cnn_channels=32, kernel_size=3,
    optimizer='adamw', lr=0.0056, weight_decay=0.0007, lr_scheduler=True,
    n_weeks=5, td_mode='context',
    include_prox=True, include_neighborhood_crowd=True,
)

# ── Folds temporales ─────────────────────────────────────────────────────────
FOLDS = [
    {'split_date': '2025-04-01', 'label': 'ene–mar / abr', 'seed': 42},
    {'split_date': '2025-05-01', 'label': 'ene–abr / may', 'seed': 123},
    {'split_date': '2025-06-01', 'label': 'ene–may / jun', 'seed': 777},
]

# Estado global compartido entre celdas
all_results    = []
baseline_stats = {}   # se rellena al terminar la celda del Baseline

In [ ]:
# ── Funciones auxiliares ──────────────────────────────────────────────────────

def bootstrap_f1(y_true, y_pred, n_boot=1000, seed=0):
    rng = np.random.default_rng(seed)
    n   = len(y_true)
    w, m = [], []
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        w.append(f1_score(y_true[idx], y_pred[idx], average='weighted', zero_division=0))
        m.append(f1_score(y_true[idx], y_pred[idx], average='macro',    zero_division=0))
    ci = lambda v: (float(np.mean(v)), float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5)))
    return {'f1_weighted': ci(w), 'f1_macro': ci(m)}


def run_one(cond_name, overrides, ablation, fold):
    params   = {**BEST_PARAMS, **overrides}
    lda_beta = params.pop('lda_beta')
    pipe = PipelineMDLF(
        nombre_almacen=ALMACEN, ttp_paths=TTP_PATHS,
        od_matrices_subdirs=OD_MATRICES_SUBDIRS, line_ap_path=LINE_PATH,
        split_date=fold['split_date'], holiday_dates=HOLIDAY_DATES,
        lda_beta=lda_beta, norm=None, n_epochs=N_EPOCHS,
        random_state=fold['seed'], ablation=ablation, **params,
    )
    pipe.run(run_name=f"{cond_name.replace(' ','_')}__fold_{fold['split_date']}")
    ci = bootstrap_f1(pipe._y_true_eval, pipe._y_pred_eval, seed=fold['seed'])
    return dict(
        condition=cond_name, fold=fold['label'],
        acc1=pipe.test_accuracy, acc5=pipe.test_accuracy5,
        f1w=ci['f1_weighted'][0], f1w_lo=ci['f1_weighted'][1], f1w_hi=ci['f1_weighted'][2],
        f1m=ci['f1_macro'][0],   f1m_lo=ci['f1_macro'][1],   f1m_hi=ci['f1_macro'][2],
    )


def run_condition(cond_name, overrides=None, ablation=None):
    """Ejecuta los 3 folds de una condición, guarda resultados e imprime resumen."""
    overrides = overrides or {}
    rows = []
    print(f"\n{'='*62}")
    print(f"  CONDICIÓN: {cond_name}")
    print(f"{'='*62}")
    for fold in FOLDS:
        print(f"\n  ▶ Fold {fold['label']}  (split={fold['split_date']}, seed={fold['seed']})")
        row = run_one(cond_name, overrides, ablation, fold)
        rows.append(row)
        all_results.append(row)
        print(f"    acc@1={row['acc1']:.4f}  acc@5={row['acc5']:.4f}  "
              f"F1w={row['f1w']:.4f} [{row['f1w_lo']:.4f},{row['f1w_hi']:.4f}]  "
              f"F1m={row['f1m']:.4f}")
    print_summary(cond_name, rows)
    return rows


def print_summary(cond_name, rows):
    """Imprime tabla de métricas medias y comparación con baseline."""
    acc1_vals = [r['acc1'] for r in rows]
    acc5_vals = [r['acc5'] for r in rows]
    f1w_vals  = [r['f1w']  for r in rows]
    f1m_vals  = [r['f1m']  for r in rows]

    mu_acc1 = np.mean(acc1_vals); std_acc1 = np.std(acc1_vals, ddof=1)
    mu_acc5 = np.mean(acc5_vals); std_acc5 = np.std(acc5_vals, ddof=1)
    mu_f1w  = np.mean(f1w_vals);  std_f1w  = np.std(f1w_vals,  ddof=1)
    mu_f1m  = np.mean(f1m_vals);  std_f1m  = np.std(f1m_vals,  ddof=1)
    # CI bootstrap agregado: promedio de los extremos de los 3 folds
    ci_lo = np.mean([r['f1w_lo'] for r in rows])
    ci_hi = np.mean([r['f1w_hi'] for r in rows])

    # Si es el baseline, guardar referencia
    if cond_name == 'Baseline (full)':
        baseline_stats.update(acc1=mu_acc1, acc5=mu_acc5, f1w=mu_f1w, f1m=mu_f1m)

    sep = '─' * 62
    print(f"\n{sep}")
    print(f"  RESUMEN — {cond_name}")
    print(sep)
    print(f"  Acc@1  : {mu_acc1:.4f} ± {std_acc1:.4f}")
    print(f"  Acc@5  : {mu_acc5:.4f} ± {std_acc5:.4f}")
    print(f"  F1-w   : {mu_f1w:.4f} ± {std_f1w:.4f}   95% CI [{ci_lo:.4f}, {ci_hi:.4f}]")
    print(f"  F1-mac : {mu_f1m:.4f} ± {std_f1m:.4f}")

    if baseline_stats and cond_name != 'Baseline (full)':
        d_acc1 = mu_acc1 - baseline_stats['acc1']
        d_f1w  = mu_f1w  - baseline_stats['f1w']
        d_f1m  = mu_f1m  - baseline_stats['f1m']
        print(sep)
        print(f"  Δ vs Baseline  Acc@1: {d_acc1:+.4f}   F1-w: {d_f1w:+.4f}   F1-mac: {d_f1m:+.4f}")
        mejor_acc1 = '✓ Baseline' if d_acc1 < 0 else '✗ Baseline peor'
        mejor_f1w  = '✓ Baseline' if d_f1w  < 0 else '✗ Baseline peor'
        print(f"  ¿Baseline mejor?  Acc@1: {mejor_acc1}   F1-w: {mejor_f1w}")
    print(sep)

In [ ]:
# ── 1. BASELINE (full) ───────────────────────────────────────────────────────
run_condition('Baseline (full)')

In [ ]:
# ── 2. −NEIGHBORHOOD ────────────────────────────────────────────────────────
run_condition('−neighborhood', overrides={'include_neighborhood_crowd': False})

In [ ]:
# ── 3. −PROX ────────────────────────────────────────────────────────────────
run_condition('−prox', overrides={'include_prox': False})

In [ ]:
# ── 4. SGD vs AdamW ─────────────────────────────────────────────────────────
run_condition('SGD', overrides={'optimizer': 'sgd'})

In [ ]:
# ── 5. −LDA ─────────────────────────────────────────────────────────────────
run_condition('−LDA', ablation={'lda'})

In [ ]:
# ── 6. −LINE ────────────────────────────────────────────────────────────────
run_condition('−LINE', ablation={'line'})

In [ ]:
# ── 7. −LDA −LINE ───────────────────────────────────────────────────────────
run_condition('−LDA −LINE', ablation={'lda', 'line'})

In [ ]:
# ── 8. −PROX −NEIGHBORHOOD ──────────────────────────────────────────────────
run_condition('−prox −neighborhood', overrides={'include_prox': False, 'include_neighborhood_crowd': False})

In [ ]:
# ── 9. −TEMPORAL (OD correlation module off) ────────────────────────────────
# Nota: el embedding temporal está integrado en el módulo OD correlation
# (atención bilineal LDA/LINE × tiempo). Ablarlo fuerza n_heads_eff=0,
# lo que desactiva el módulo completo. Es el bloque con mayor importancia
# por atribución de gradiente en el baseline (temporal=0.1287).
run_condition('−temporal', ablation={'temporal'})

In [ ]:
# ── 10. −INDIVIDUAL ─────────────────────────────────────────────────────────
run_condition('−individual', ablation={'individual'})

In [ ]:
# ── 11. −CROWD (incluye −neighborhood por dependencia técnica) ───────────────
# NeighborhoodCrowdExtractor depende de self._hist_crowd que solo existe
# si _extract_crowd_features() se ejecuta → al ablacionar crowd, neighborhood
# también se desactiva automáticamente.
run_condition('−crowd', ablation={'crowd'})

In [ ]:
# ── 12. −PROX −TEMPORAL ─────────────────────────────────────────────────────
# MOTIVACIÓN: −prox (−0.86pp) y −temporal (−1.10pp) tienen magnitudes similares,
# lo que sugiere que ambos módulos podrían estar capturando el mismo fenómeno
# por caminos distintos:
#   · −temporal (OD module): señal POBLACIONAL — flujos zona-origen × zona-destino × tiempo
#   · −prox: señal INDIVIDUAL — historial del usuario en estaciones vecinas del destino
# En cercanías ambas responden a la misma pregunta de fondo (patrones de commuting).
#
# Si son independientes → Δ(−prox −temporal) ≈ −1.10 + (−0.86) = −1.96pp
# Si son redundantes    → Δ(−prox −temporal) ≈ −1.10pp  (prox no añade nada sin OD)
# Si hay solapamiento   → Δ(−prox −temporal) ∈ (−1.10, −1.96)  [esperado]
#
# El resultado desambigua si prox aporta señal propia o es un proxy del módulo OD.
run_condition('−prox −temporal', ablation={'temporal'}, overrides={'include_prox': False})

In [ ]:
# ── TABLA FINAL COMPLETA ─────────────────────────────────────────────────────
df = pd.DataFrame(all_results)
df.to_csv(_mdlf_dir / 'validacion_temporal_results.csv', index=False)

order = ['Baseline (full)', '−neighborhood', '−prox', 'SGD', '−LDA', '−LINE', '−LDA −LINE', '−prox −neighborhood', '−temporal', '−individual', '−crowd', '−prox −temporal']
rows  = []
for cond in order:
    g = df[df['condition'] == cond]
    if g.empty: continue
    mu_acc1 = g['acc1'].mean(); std_acc1 = g['acc1'].std(ddof=1)
    mu_f1w  = g['f1w'].mean();  std_f1w  = g['f1w'].std(ddof=1)
    mu_f1m  = g['f1m'].mean();  std_f1m  = g['f1m'].std(ddof=1)
    ci_lo   = g['f1w_lo'].mean(); ci_hi  = g['f1w_hi'].mean()
    d_acc1  = mu_acc1 - baseline_stats['acc1']
    d_f1w   = mu_f1w  - baseline_stats['f1w']
    rows.append(dict(
        Condición=cond,
        **{'Acc@1': f"{mu_acc1:.4f}±{std_acc1:.4f}",
           'ΔAcc@1': '' if cond == 'Baseline (full)' else f"{d_acc1:+.4f}",
           'F1-weighted': f"{mu_f1w:.4f}±{std_f1w:.4f}",
           '95% CI': f"[{ci_lo:.4f},{ci_hi:.4f}]",
           'ΔF1-w': '' if cond == 'Baseline (full)' else f"{d_f1w:+.4f}",
           'F1-macro': f"{mu_f1m:.4f}±{std_f1m:.4f}"},
    ))

table = pd.DataFrame(rows).set_index('Condición')
print(table.to_string())
table